## Start here — Google Colab

Open this notebook from the course website or GitHub. On first use, the setup cell connects Google Drive and places the public course folder there if it is not already present. It expands the prepared synthetic inputs and installs the pinned libraries automatically.

Start with **01** and run each notebook in order: **01 → 02 → 03 → 04**. Keep the course folder together and use the same Colab runtime for the sequence. CPU is sufficient. The data-generation notebook is not part of this learner path.

In [ ]:
# Connect Drive and prepare this course automatically before importing lesson libraries.
from pathlib import Path
import gzip
import os
import shutil
import sys
import json
import subprocess
import importlib.metadata as package_metadata

COURSE_ID = 'ground-truth-gym-l3-4.0'
MARKER = 'COURSE_PACKAGE.json'
REPOSITORY_URL = 'https://github.com/GroundTruthML/Course_Material.git'
REPOSITORY_FOLDER = 'GroundTruth_Course_Material'
try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
    search_root = Path('/content/drive/MyDrive')
else:
    search_root = Path.cwd()


def is_course_folder(path):
    marker = path / MARKER
    return marker.is_file() and json.loads(marker.read_text()).get('course_id') == COURSE_ID


def find_local_course():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if is_course_folder(candidate):
            return candidate
    return None


def find_drive_course(root):
    matches = []
    for directory, subfolders, files in os.walk(root):
        subfolders[:] = [name for name in subfolders if not name.startswith('.') and name not in {'node_modules', '__pycache__', '.git'}]
        if MARKER in files:
            candidate = Path(directory)
            if is_course_folder(candidate):
                matches.append(candidate)
                subfolders[:] = []
    canonical = root / REPOSITORY_FOLDER / 'course_material' / 'gym'
    if is_course_folder(canonical):
        return canonical
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError('More than one Gym course folder was found in My Drive. Keep one copy, then rerun this cell.')
    return None


course_root = find_drive_course(search_root) if drive is not None else find_local_course()
if course_root is None and drive is not None:
    repository_root = search_root / REPOSITORY_FOLDER
    if repository_root.exists():
        raise RuntimeError('The course repository folder exists but is incomplete. Remove that incomplete copy from My Drive and rerun this cell.')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(repository_root)])
    course_root = repository_root / 'course_material' / 'gym'
if course_root is None or not is_course_folder(course_root):
    raise RuntimeError('Open this notebook from the public Course_Material repository, or keep the complete course_material/gym folder together.')
os.chdir(course_root)

# Expand only the prepared synthetic input files. Notebook outputs are written separately.
DATA_DIR = course_root / 'gym_case_data'
INPUT_DIR = course_root / 'input_data' / 'gym_case_data'
if not (DATA_DIR / 'manifest.json').exists():
    shutil.copy2(INPUT_DIR / 'manifest.json', DATA_DIR / 'manifest.json')
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
(DATA_DIR / 'raw').mkdir(parents=True, exist_ok=True)
for table_name in manifest['tables']:
    csv_path = DATA_DIR / 'raw' / f'{table_name}.csv'
    if not csv_path.exists():
        compressed_path = INPUT_DIR / 'raw' / f'{table_name}.csv.gz'
        if not compressed_path.exists():
            raise FileNotFoundError(f'Missing prepared input: {compressed_path}')
        with gzip.open(compressed_path, 'rb') as source, csv_path.open('wb') as destination:
            shutil.copyfileobj(source, destination)

requirements = (course_root / 'requirements-colab.txt').read_text().splitlines()
requirements = [line.strip() for line in requirements if line.strip() and not line.startswith('#')]
if drive is not None:
    dependency_dir = Path('/content/groundtruth_gym_dependencies')
    stamp = dependency_dir / 'course_requirements.json'
    if not stamp.exists() or json.loads(stamp.read_text()) != requirements:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--target', str(dependency_dir), *requirements])
        stamp.write_text(json.dumps(requirements))
    sys.path.insert(0, str(dependency_dir))
    for distribution, module in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('scikit-learn', 'sklearn'), ('matplotlib', 'matplotlib'), ('joblib', 'joblib')]:
        expected = next(line.split('==')[1] for line in requirements if line.startswith(distribution + '=='))
        loaded = sys.modules.get(module)
        if loaded is not None and getattr(loaded, '__version__', expected) != expected:
            raise RuntimeError('Please select Runtime > Disconnect and delete runtime, reconnect, and Run all. No path or package configuration is needed.')

for item in requirements:
    distribution, expected = item.split('==')
    actual = package_metadata.version(distribution)
    if actual != expected:
        raise RuntimeError(f'{distribution}: expected {expected}, found {actual}. Use a fresh Colab runtime and run this first cell.')

for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[variable] = '1'
print('Course ready:', course_root)
print('Prepared synthetic inputs are ready. Run notebooks 01 → 02 → 03 → 04 in this order.')

# 04 — Score members, propose lists and make a business recommendation

**Purpose:** Reload03's fitted pipeline, score 1,200 upcoming eligible members and propose a 1,000-person list alongside the staff rule.

| Section | Main output |
|---|---|
| ACT-01 | Reproducible purchase scores |
| ACT-02 | Two review-only promotion lists and overlap/store comparison |
| ACT-04 | Evidence-led business recommendation |
| REF-01 | Staff-support/automation reflection; no additional model calculation |

**Inputs:** trusted model_bundle.joblib, model_card.json, member_features.csv and comparison reports. **Outputs:** scores, proposed lists and business_recommendation.json.

**Reading pattern:** question → plan → annotated code → output → takeaway. Scoring reuses learned state; these outputs are proposals rather than an executed campaign. ACT-03 was removed; stable section IDs are retained.

## ACT-01 — Use the saved training state on upcoming members

### The question
Can we score upcoming members with the same learned preparation?

### The plan
- Verify the artifact hash/version and required inputs.
- Select all 1,200 eligible upcoming members with unknown outcomes.
- Check engineered formulas and valid numerical measurements.
- Use saved transform/predict only; verify reload consistency.

### What to look for
Inspect score range, representation across PT groups and categories unseen in training.

In [ ]:
# Import libraries for saved files, arrays, tables and inspection.
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import sklearn
import joblib


# Input/output paths: notebooks run from Current beside gym_case_data.
DATA_DIR = Path('gym_case_data')


# Locate the saved fitted model separately from proposed campaign outputs.
MODEL_DIR = DATA_DIR / 'model'


# Write scores/lists/recommendation under actions; no real campaign execution here.
ACTION_DIR = DATA_DIR / 'actions'
ACTION_DIR.mkdir(exist_ok=True)


# Business contact cutoff; booking capacity is separate context, not a ranking constraint.
CONTACT_LIMIT = 1000
BOOKING_LIMIT = 200


# Check the preceding notebook artifact exists before continuing.
assert (MODEL_DIR / 'model_bundle.joblib').exists(), 'Run notebook 03 first.'


# Record rules and file hashes so01 can verify the exact fixture.
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())


# Verify model bytes and library version before loading the artifact.
model_card = json.loads((MODEL_DIR / 'model_card.json').read_text())
assert hashlib.sha256((MODEL_DIR / 'model_bundle.joblib').read_bytes()).hexdigest() == model_card['artifact_sha256']
assert model_card['sklearn_version'] == sklearn.__version__, 'Use the training library version before loading this artifact.'


# Load training-fitted preprocessing and estimator, without fitting upcoming members.
bundle = joblib.load(MODEL_DIR / 'model_bundle.joblib')
assert bundle['data_version'] == manifest['data_version']


# Load member-level measurements; parse dates before temporal assertions.
features = pd.read_csv(DATA_DIR / 'member_features.csv')
for column in ['decision_time', 'signup_at']:
    features[column] = pd.to_datetime(features[column], utc=True, format="mixed")


# Reserve future outcomes; use unlabeled profiles only for population/coverage or scoring.
upcoming = features.loc[features.campaign_id.eq('upcoming')].copy()


# Check eligibility, unique identities, unknown outcomes and required feature columns.
assert len(upcoming) == 1200 and upcoming.member_id.is_unique
assert upcoming.eligible.eq(True).all()
assert set(upcoming.pt_relationship) == {'never_pt', 'active_pt', 'lapsed_pt'}
assert upcoming.label.isna().all()
assert set(bundle['feature_columns']).issubset(upcoming.columns)


# Verify saved engineered inputs follow the recorded fixed formulas.
if 'log_checkins_30d' in bundle['feature_columns']:

    # Check reload equivalence or engineered-formula consistency as specified below.
    np.testing.assert_allclose(upcoming.log_checkins_30d, np.log1p(upcoming.checkins_30d))
if 'visit_share_7d' in bundle['feature_columns']:
    expected_share = np.divide(upcoming.checkins_7d, upcoming.checkins_30d, out=np.zeros(len(upcoming)), where=upcoming.checkins_30d.to_numpy()>0)

    # Check reload equivalence or engineered-formula consistency as specified below.
    np.testing.assert_allclose(upcoming.visit_share_7d, expected_share)
assert upcoming.decision_time.le(pd.Timestamp(manifest['extracted_at'])).all()


# Validate finite measurements; momentum may legitimately be negative.
for column in bundle['numeric_features']:
    assert np.isfinite(upcoming[column].dropna()).all()

    # Momentum is a signed rate difference: decreasing activity legitimately produces a negative value.
    if column != 'checkin_momentum':
        assert upcoming[column].dropna().ge(0).all()
print('[INFO] Candidate:', bundle['selected_candidate'], '| contacts', CONTACT_LIMIT, '| bookings', BOOKING_LIMIT)
print('[INFO] Target:', bundle['target'])


# Retain traceable audit fields alongside each new purchase score.
scored = upcoming[['member_id', 'location_id', 'decision_time', 'signup_at', 'inquiry_60d', 'pt_relationship', 'prior_pt_purchases', 'checkins_30d', 'checkin_momentum']].copy()


# predict_proba[:,1] gives purchase probability from the saved complete pipeline.
scored['purchase_score'] = bundle['pipeline'].predict_proba(upcoming[bundle['feature_columns']])[:, 1]
assert scored.purchase_score.between(0, 1).all()
assert len(scored)==1200 and scored.member_id.is_unique
display(scored.groupby('pt_relationship').agg(members=('member_id','size'), mean_score=('purchase_score','mean')))


# Do not force equal group selection: report representation, but never prefilter by PT history.
# Reloading the same saved pipeline must not change the member scores.
reloaded = joblib.load(MODEL_DIR / 'model_bundle.joblib')


# Check reload equivalence or engineered-formula consistency as specified below.
np.testing.assert_allclose(scored.purchase_score, reloaded['pipeline'].predict_proba(upcoming[bundle['feature_columns']])[:, 1])


# Save the scored population before applying any business cutoff.
scored.to_csv(ACTION_DIR / 'upcoming_scores.csv', index=False)
display(scored.head())


# Inspect new categorical values handled by the existing encoding vocabulary.
unseen_counts = {}
if bundle['categorical_features']:
    encoder = bundle['pipeline'].named_steps['features'].named_transformers_['categorical'].named_steps['encode']

    # Inspect new categorical values handled by the existing encoding vocabulary.
    unseen_counts = {column: int((upcoming[column].notna() & ~upcoming[column].isin(categories)).sum())
                     for column, categories in zip(bundle['categorical_features'], encoder.categories_)}
print('[WARN] Members with categories unseen during training:', unseen_counts)

### Takeaway — ACT-01

**All1,200 members have reproducible scores.** Their future outcomes were not used for learning, selection or evaluation.

**Before moving on:** Explain how this output supports the next decision.

## ACT-02 — Translate scores into a proposed promotion list

### The question
How does the business budget turn scores into a reference list?

### The plan
- Rank by purchase probability with member-ID ties.
- Take the business contact limit of1,000.
- Generate the staff-rule list from the same eligible population.
- Compare list overlap and store representation; export review-only lists.

### What to look for
A changed contact budget changes the cutoff; evaluate the new use even when the saved model stays the same.

In [ ]:
# Apply model probability ordering and the business contact cutoff.
model_list = scored.sort_values(['purchase_score', 'member_id'], ascending=[False, True]).head(CONTACT_LIMIT).copy()


# Build the current staff-rule list from the same eligible member pool.
rule_list = scored.sort_values(['inquiry_60d', 'signup_at', 'member_id'], ascending=[False, True, True]).head(CONTACT_LIMIT).copy()


# Label both outputs as review-only proposals and validate equal list sizes.
model_list['proposal_status'] = 'review_only_not_sent'
rule_list['proposal_status'] = 'baseline_review_only_not_sent'
assert len(model_list) == len(rule_list) == CONTACT_LIMIT
assert model_list.member_id.is_unique and rule_list.member_id.is_unique


# Export concrete model and rule proposals for review.
model_list.to_csv(ACTION_DIR / 'proposed_model_promotion_list.csv', index=False)
rule_list.to_csv(ACTION_DIR / 'baseline_promotion_list.csv', index=False)


# Count common members; differences describe proposed selection changes, not extra purchases.
overlap = len(set(model_list.member_id) & set(rule_list.member_id))
print(f'[RESULT] Proposed list: {len(model_list)} of {len(upcoming)} eligible members; {overlap} shared with the rule.')


# Compare store representation under both proposals.
store_counts = pd.concat([model_list.assign(policy='Model proposal'), rule_list.assign(policy='Staff rule')]).groupby(['location_id', 'policy']).size().unstack(fill_value=0)
display(store_counts)
fig, ax = plt.subplots(figsize=(10, 5))
store_counts.plot.bar(ax=ax, rot=0, color=['#2166AC', '#6B7280'])
ax.set(title='How does the proposed list change store representation?', xlabel='Store', ylabel='Selected promotion attempts')
fig.tight_layout(); fig.savefig(ACTION_DIR / 'proposed_list_by_store.png', dpi=150); plt.show(); plt.close(fig)

### Takeaway — ACT-02

**The model supplies scores; the business chooses the cutoff and action.** The two exported lists show proposed choices, not measured upcoming sales.

**Before moving on:** Explain how this output supports the next decision.

## ACT-04 — Decide whether to replace the staff rule

### The question
Should the gym replace its current staff rule?

### The plan
- Reload validation and April reports.
- Use the descriptive screen and known limitations to form a recommendation.
- Save decision context, evidence needs and responsible roles.

### What to look for
Compare validation35/31 and April16/18 within each cohort. These are equally sized lists at the same promotion fraction.

In [ ]:
# Reload the actual comparisons before recommending a business action.
validation_report = pd.read_csv(MODEL_DIR / 'validation_comparison.csv')
test_report = pd.read_csv(MODEL_DIR / 'test_comparison.csv')
display(validation_report); display(test_report)


# Translate the descriptive screen into a bounded evidence-based recommendation.
recommendation = ('Consider a bounded prospective policy evaluation after reviewing data coverage and operational readiness.'
                  if bundle['offline_screen_passed'] else
                  'Retain the staff rule; this offline comparison does not justify replacement. Review data coverage and model limitations.')


# Record decision context, limitations and next owners alongside the recommendation.
recommendation_record = {'data_version': bundle['data_version'], 'candidate': bundle['selected_candidate'],
                         'eligible_members': len(upcoming), 'proposed_contacts': len(model_list),
                         'booking_limit': BOOKING_LIMIT, 'queue_rule': 'first come; close after 200 accepted bookings',
                         'offline_screen_passed': bundle['offline_screen_passed'], 'test_status': bundle['test_status'], 'recommendation': recommendation,
                         'approval_to_send': False, 'incremental_business_value_measured': False,
                         'model_limitations': bundle['limitations'],
                         'next_owners': {'data_owner': 'resolve missing payment coverage',
                                         'operations': 'confirm delivery, booking and consultation recording',
                                         'analyst': 'design independent policy evaluation with capacity accounted for'}}


# Save the recommendation; actual promotion approval/execution remains separate.
(ACTION_DIR / 'business_recommendation.json').write_text(json.dumps(recommendation_record, indent=2))
print('[RESULT]', recommendation)
print('[WARN] All exported lists are review-only. No contacts, bookings or purchases were executed.')

### Takeaway — ACT-04

**Retain the current staff rule and preserve model scores as reference outputs.** Further decisions should follow additional outcome/coverage evidence.

**Before moving on:** Explain how this output supports the next decision.

## REF-01 — Support staff decisions with a repeatable workflow

### What does the comparison leave us with?

After all that modeling, the comparison led us to keep the staff rule.

So… do we close the notebook, quietly rename the project “learning experience,” and move on?

Not quite. We tested whether the model could make a better selection. There is another useful question: could this work make the staff’s job easier?

### Where could the workflow help?

Think about preparing a normal campaign. Someone gathers recent activity, checks PT history, and puts together a list. Staff then add what they know from talking to members every day.

The data might say, “This member looks promising.” A staff member might say, “Yes, but she’s moving to another city next week.”

That is useful information. It just hasn’t made its way into our feature matrix.

So the system could handle the repeatable preparation: collect the measurements, apply the rule, and produce a reference list. Staff can spend their time reviewing the people who need attention.

If the model suggests someone different, staff can inspect the available evidence and decide whether the suggestion helps. We record those changes so we can learn what the tool contributes.

### Compare three working processes

But before we announce that we have “transformed the workflow”—a phrase that looks excellent on a slide—we should compare three ways of doing the work.

First, the existing manual preparation. Second, automated preparation using the current staff rule. Third, that same automated process with model scores available as an additional reference.

Why compare all three? Because a faster spreadsheet pipeline and a useful prediction model deserve separate credit. Otherwise, the model might collect an award for work the automation did.

### Measure the opportunity

We measure preparation time and review time. We check how often staff change the suggestions, and why. Then we follow purchase outcomes for the resulting lists. If we change who receives the promotion, we plan a campaign comparison to assess that change.

### What would the results mean?

Several results are possible.

Rule automation might save time while model scores add very little. That could still be worth building.

Model scores might help staff notice useful exceptions. That gives prediction a practical role.

Or the scores might create more review work than they save. A tool that needs its own full-time babysitter has a harder business case.

So we ask two questions: does this process support better decisions, and does it make the work faster and more consistent?

Those benefits are worth measuring separately. We have not demonstrated them yet; we have identified a practical opportunity to test.

### Carry the lesson into the next project

That is the lesson to carry into the next project: understand the real work, automate the repeatable preparation, and use evidence to decide where the model belongs.

Sometimes the most useful thing your model can do is help an experienced person get through the list before lunch.

### Takeaway — REF-01
**Compare the human workflow, rule automation and model-assisted review.** Measure preparation/review effort and decision evidence before assigning the model a business role.